In [ ]:
import torch
import torch.nn as nn 
from torch.nn import functional as F
import pickle

#device = 'cuda' if torch.cuda.is_available() else 'cpu'

block_size = 18
batch_size = 24
max_iters = 3000
learning_rate = 3e-3
eval_iters = 100
n_layer = 4
n_embd = 384  #embdedding dim 
n_head = 4 #no. of heads running in parallel 
dropout = 0.2

In [ ]:
chars = ""
with open('wizard_of_oz.txt', 'r', encoding='utf-8') as file:
    text = file.read()
    chars = sorted(set(text))

vocab_size = len(chars)

In [ ]:
string_to_int = {char:i for i,char in enumerate(chars)}
int_to_string = {i:char for i,char in enumerate(chars)}

encode = lambda st : [string_to_int[ch] for ch in st]
decode = lambda l: ''.join([int_to_string[i] for i in l])

data = torch.tensor(encode(text), dtype = torch.long)

In [ ]:
n = int(0.8*len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x,y

In [ ]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [ ]:
class Head(nn.Module):  #scaled dot product attention
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias = False)
        self.query =  nn.Linear(n_embd, head_size, bias = False)
        self.value =  nn.Linear(n_embd, head_size, bias = False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        B ,T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        #compute attention scores
        wei = q @ k.transpose(-2, -1) * k.shape[-1]**-0.5 #[B, T, hs] @[B, hs, T] -> [B, T, T]
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim = -1)
        wei = self.dropout(wei)
        #perform weighted aggregation of the values
        v = self.value(x)
        out = wei @ v
        return out

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim = -1)
        out = self.dropout(self.proj(out))
        return out
    

class FeedForward(nn.Module):

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4* n_embd, n_embd),
            nn.Dropout(dropout),
        )
    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    """Transformer block: communicataion followed by computation"""

    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head  #how many faetures is a single head capturing during multihead attention
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        y = self.sa(x)
        x = self.ln1(x+y)
        y = self.ffwd(x)
        x = self.ln2(x + y)
        return x

class GPTLanguageModel(nn.Module):      #creating the GPT model class
    def __init__(self, vocab_size):
        super().__init__()      #inherit the nn.module parent class using the super keyword
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)  #create 1D n_embd size vectors for each of our 78 chars 
        self.position_embedding_table  = nn.Embedding(block_size, n_embd)    #create 1D n_embd size vectors for each our chars in block_size  (for each letter index)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])    #create four sequential decoder layers using nn.sequential
        self.ln_f = nn.LayerNorm(n_embd) #final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)
        
# EXP STD TO SET STANDARD DEVIATION FOR WEIGHTS
        
        self.apply(self._init_weights)  ## initalises our weights around certain std deviations to avoid vanishing\exploding gradients and also making sure that all neurons dont have the same weights ##

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None: 
                torch.nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Embedding):
                torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)



    def forward(self, index, targets = None): 
        #allocate fixed positional encodings based on sin and cos functions to identify the sequence a particular 
        #sentence follows, how far one word is from another. 
        #idx and targets are (B,T) tensor of integers 
        
        B, T = index.shape

        tok_emb = self.token_embedding_table(index)  #(B, T, C)
        pos_emb = self.position_embedding_table(torch.arange(T)) #(T, C)   #device = device
        x = tok_emb + pos_emb
        x = self.blocks(x) #feeding the embeddings to the sequential network
        x = self.ln_f(x)
        logits = self.lm_head(x)
        
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape          #B = batch_size, T= no of integers or block_size , C = vocab_size
            logits = logits.view(B*T, C)    #adjust the dim of the tensor as per loss function requirement
            targets = targets.view(B*T)     #adjust the target dim as per loss function requirement
            loss = F.cross_entropy(logits, targets) #calculate loss with target values and logits 
            
        return logits, loss 
    
    def generate(self, index, max_new_tokens):
   # index is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            index_cond = index[:, -block_size:]
            # get the predictions
            logits, loss = self.forward(index_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            index_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            index = torch.cat((index, index_next), dim=1) # (B, T+1)
        return index

model = GPTLanguageModel(vocab_size)
with open('model-01.pkl', 'rb') as f:
    model = pickle.load(f)
print('done')


In [ ]:
optimiser = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):
    if iter % eval_iters == 0:
        losses = estimate_loss()
        print(f'step: {iter}, loss: {losses}')
    
    xb, yb = get_batch('train') #sample a batch of data
    
    logits, loss = model.forward(xb, yb) #call the .forward method with that sample
    optimiser.zero_grad(set_to_none = True)
    loss.backward()
    optimiser.step()
print(loss.item())    

with open('model-01.pkl', 'wb') as f:
    pickle.dump(model, f)